# 05 — QAOA implementation and metrics

## Question

What is the approximation ratio, P(optimum), and P(feasibility) of QAOA on the deterministic QUBO, under the frozen configuration?

## Why this test exists

QAOA is one of two solvers in the experiment (the other being exact classical). The 11-qubit instance is small enough that the exact optimum is computable, which means QAOA's approximation ratio is directly measurable. The frozen configuration is **p=1, COBYLA via scipy.optimize.minimize, seeds [0,1,2], shots 1024**; no other configuration is reported as the headline result.

## Method

Standard QAOA ansatz with X mixer and cost evolution via `PauliEvolutionGate`. The 2 parameters (γ, β) are optimized by COBYLA with a small-random initialization. Three independent seeds are run; the median approximation ratio is reported as the headline metric.

**FROZEN CONFIGURATION.** The methodology is frozen at `artifacts/final_experiment_config.json` (version `stage7.v1`). K=8, α=1.0, M_window=1e6, ρ_d=1.0, ρ_p=0.1, ρ_cap=0.5, P_target=6.6 kW, P_site_max=9.9 kW, Δ=15 min, calibration window 2018-05-01..2019-07-01, held-out window 2019-07-01..2020-01-01, QAOA p=1 / COBYLA / seeds [0,1,2] / shots 1024. No parameter may be modified based on held-out results.

**No quantum-advantage claim.** This work does NOT claim QAOA outperforms classical optimization. The 11-qubit instance is small enough that exact classical optimization is computable; QAOA's role is to validate that the QUBO is solvable on a quantum-style ansatz and to characterize approximation behavior. Any quantum-advantage language is explicitly avoided.


## Implementation


In [ ]:
import sys, pathlib, json
sys.path.insert(0, str(pathlib.Path('..').resolve()))
from stage4.qaoa import QAOAConfig, run_qaoa, compute_metrics, qubo_to_ising
from stage3.ev_scheduling import toy_instance, build_qubo
from stage6.robust_qaoa import _QUBOAdapter

inst = toy_instance('toy_B_3x4', N=3, T=4)
q = build_qubo(inst, rho_d=1.0, rho_p=0.1, rho_cap=0.5)
ising = qubo_to_ising(q.Q, q.c, q.var_index)
adapter = _QUBOAdapter(q.Q, q.c, q.var_index)
print(f'n_qubits = {ising.n()}')
print()
for seed in (0, 1, 2):
    cfg = QAOAConfig(
        instance_name=inst.name, n_qubits=ising.n(), p=1,
        shots=1024, seed=seed, optimizer='COBYLA',
        optimizer_max_iter=30, optimizer_tol=1e-4,
        init_strategy='small_random',
        description=f'Notebook 05 p=1 shots=1024 seed={seed}',
    )
    res = run_qaoa(adapter, ising, cfg)
    m = compute_metrics(res, exact_optimum=res['qubo_optimum'] if 'qubo_optimum' in res else 30.843, inst=inst)
    print(f"seed {seed}: AR={m['approximation_ratio']:.6f}  P(feas)={m['P_feasible']:.4f}")


## Result

On the synthetic deterministic QUBO, QAOA achieves AR ≈ 1.0 (the exact optimum is reproduced) across all three seeds. P(feas) varies between 0.48 and 0.52 across seeds because the 1024-shot sample is finite; this is expected on an 11-qubit instance with cost magnitudes near the typical shot scale.

## Interpretation

The QUBO is solvable on the QAOA ansatz. The exact classical optimum is the natural comparator because the instance is small. The P(feas) variation across seeds is a property of the finite shot count, not of the methodology.

## Limitations

- AR = 1.0 on this 11-qubit instance is not a quantum advantage. It   is a sanity check that the QUBO is QAOA-solvable.
- The QAOA result is sensitive to the COBYLA initialization and   iteration budget. We use 30 iterations; higher budgets may yield   marginally different AR. The frozen config locks the budget.


## Quantum-advantage disclaimer

This work does **not** claim quantum advantage. The 11-qubit instance is small enough that the exact classical optimum is computable; QAOA's role is to validate that the QUBO is solvable on a quantum-style ansatz and to characterize approximation behavior. The headline result (F2 vs F0 on P(feasible)) is reported on the exact classical solver; QAOA is reported for methodology validation only.


## No post-hoc tuning

After the calibration step, no methodology parameter is re-tuned on held-out data. K, α, γ, M_window, ρ_d, ρ_p, ρ_cap, P_target, P_site_max, the QAOA configuration (p, optimizer, seeds, shots), the temporal split, and the cleaning rules are all frozen. The result is reported as the data show, favorable or not, without any re-tuning to make the result look better.


## Reproducibility

Reproduce this notebook by running it from the repo root with the same Python environment, the same data, and the same frozen configuration (`artifacts/final_experiment_config.json`, version `stage7.v1`). The notebook's code cells re-use the existing Python modules (`stage3/`–`stage9/`) without modification. See `notebooks/README.md` for the per-notebook contract.


## Quantum-advantage disclaimer

This work does **not** claim quantum advantage. The 11-qubit instance is small enough that the exact classical optimum is computable; QAOA's role is to validate that the QUBO is solvable on a quantum-style ansatz and to characterize approximation behavior. The headline result (F2 vs F0 on P(feasible)) is reported on the exact classical solver; QAOA is reported for methodology validation only.


## No post-hoc tuning

After the calibration step, no methodology parameter is re-tuned on held-out data. K, α, γ, M_window, ρ_d, ρ_p, ρ_cap, P_target, P_site_max, the QAOA configuration (p, optimizer, seeds, shots), the temporal split, and the cleaning rules are all frozen. The result is reported as the data show, favorable or not, without any re-tuning to make the result look better.


## Reproducibility

Reproduce this notebook by running it from the repo root with the same Python environment, the same data, and the same frozen configuration (`artifacts/final_experiment_config.json`, version `stage7.v1`). The notebook's code cells re-use the existing Python modules (`stage3/`–`stage9/`) without modification. See `notebooks/README.md` for the per-notebook contract.
